# Chapter 2: Frame Definability

This notebook follows Chapter 2 of [Boxes and Diamonds](https://bd.openlogicproject.org)
using the **Gamen.jl** package.

We cover:
- Validity on a frame (Definition 2.1)
- Frame properties: reflexivity, symmetry, transitivity, seriality, euclideanness (Definition 2.3)
- The correspondence between modal schemas and frame properties

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..", ".."))
using Gamen

### Unicode Modal Operators

Gamen.jl exports `□` and `◇` as Unicode aliases for `Box` and `Diamond`. Type `\square<tab>` and `\diamond<tab>` in the Julia REPL. Throughout this notebook, we use these Unicode operators to write formulas that mirror the mathematical notation.

## 2.1 Validity on a Frame

Recall from Chapter 1 that a formula can be *true in a model*. But a model
includes a specific valuation $V$. A stronger notion is *validity on a frame*:

> **Definition 2.1.** A formula $A$ is *valid on a frame* $F = \langle W, R \rangle$
> if $A$ is true in every model $M = \langle W, R, V \rangle$ based on $F$ —
> that is, for every possible valuation $V$.

This tells us what a frame's *structure* (its accessibility relation) forces
to be true, independent of which propositions hold where.

In [ ]:
p = Atom(:p)
q = Atom(:q)

frame = KripkeFrame([:w1, :w2], [:w1 => :w2])

# □⊤ is valid on any frame — ⊤ is true everywhere, so □⊤ is too
# Using Unicode: □ === Box, ◇ === Diamond
println("□⊤ valid on frame: ", is_valid_on_frame(frame, □(Top())))

In [ ]:
# But □p → p is NOT valid on this frame — it's not reflexive
println("□p → p valid: ", is_valid_on_frame(frame, Implies(□(p), p)))

## 2.2 Frame Properties (Definition 2.3)

The key insight of frame definability is that certain *structural properties*
of the accessibility relation correspond to specific modal *schemas*.

| Property | Condition | Intuition |
|:---------|:----------|:----------|
| Reflexive | $\forall w.\; Rww$ | Every world accesses itself |
| Symmetric | $\forall w, w'.\; Rww' \to Rw'w$ | Access goes both ways |
| Transitive | $\forall w, w', w''.\; Rww' \land Rw'w'' \to Rww''$ | Access chains compose |
| Serial | $\forall w.\; \exists w'.\; Rww'$ | Every world has a successor |
| Euclidean | $\forall w, w', w''.\; Rww' \land Rww'' \to Rw'w''$ | Successors see each other |

### Testing Frame Properties

Let's build some frames and check their properties:

In [ ]:
# A reflexive, transitive frame (a preorder)
preorder = KripkeFrame([:w1, :w2, :w3],
    [:w1 => :w1, :w2 => :w2, :w3 => :w3,   # reflexive loops
     :w1 => :w2, :w2 => :w3, :w1 => :w3])   # transitive chain

println("Preorder frame:")
println("  reflexive:  ", is_reflexive(preorder))
println("  symmetric:  ", is_symmetric(preorder))
println("  transitive: ", is_transitive(preorder))
println("  serial:     ", is_serial(preorder))
println("  euclidean:  ", is_euclidean(preorder))

In [ ]:
# An equivalence relation (reflexive + symmetric + transitive)
equiv = KripkeFrame([:w1, :w2],
    [:w1 => :w1, :w2 => :w2, :w1 => :w2, :w2 => :w1])

println("Equivalence frame:")
println("  reflexive:  ", is_reflexive(equiv))
println("  symmetric:  ", is_symmetric(equiv))
println("  transitive: ", is_transitive(equiv))
println("  serial:     ", is_serial(equiv))
println("  euclidean:  ", is_euclidean(equiv))

## 2.3 The Correspondence Results

The central results of Chapter 2 show that each frame property corresponds
to a modal schema being valid on the frame:

| Schema | Name | Formula | Frame Property |
|:-------|:-----|:--------|:---------------|
| **K** | Distribution | $\square(p \to q) \to (\square p \to \square q)$ | *All frames* |
| **T** | Reflexivity | $\square p \to p$ | Reflexive |
| **D** | Seriality | $\square p \to \diamond p$ | Serial |
| **B** | Symmetry | $p \to \square\diamond p$ | Symmetric |
| **4** | Transitivity | $\square p \to \square\square p$ | Transitive |
| **5** | Euclideanness | $\diamond p \to \square\diamond p$ | Euclidean |

Let's verify each one.

### Schema K: Valid on All Frames (Proposition 1.19)

$\square(p \to q) \to (\square p \to \square q)$

This is the *distribution axiom* — it holds on every frame, regardless of
structure. It says that $\square$ distributes over implication.

In [ ]:
# □(p → q) → (□p → □q), using Unicode □
schema_k = Implies(□(Implies(p, q)), Implies(□(p), □(q)))
println("Schema K: ", schema_k)

frame1 = KripkeFrame([:w1, :w2], [:w1 => :w2])
frame2 = KripkeFrame([:w1, :w2, :w3], [:w1 => :w2, :w2 => :w3])
frame3 = KripkeFrame([:w1], [:w1 => :w1])

println("Valid on frame1: ", is_valid_on_frame(frame1, schema_k))
println("Valid on frame2: ", is_valid_on_frame(frame2, schema_k))
println("Valid on frame3: ", is_valid_on_frame(frame3, schema_k))

### Schema T: □p → p corresponds to Reflexivity (Proposition 2.5)

If every world can see itself, then whatever is necessary is actual.
Conversely, if $\square p \to p$ is valid, the frame must be reflexive.

In [ ]:
schema_t = Implies(□(p), p)
println("Schema T: ", schema_t)

reflexive_frame = KripkeFrame([:w1, :w2],
    [:w1 => :w1, :w1 => :w2, :w2 => :w2])
non_reflexive_frame = KripkeFrame([:w1, :w2], [:w1 => :w2])

println("Reflexive frame:     ", is_valid_on_frame(reflexive_frame, schema_t))
println("Non-reflexive frame: ", is_valid_on_frame(non_reflexive_frame, schema_t))

### Schema D: □p → ◇p corresponds to Seriality (Proposition 2.7)

If every world has at least one successor, then whatever is necessary
is at least possible. A world with no successors makes $\square p$
vacuously true while $\diamond p$ is false.

In [ ]:
schema_d = Implies(□(p), ◇(p))
println("Schema D: ", schema_d)

serial_frame = KripkeFrame([:w1, :w2], [:w1 => :w2, :w2 => :w1])
non_serial_frame = KripkeFrame([:w1, :w2], [:w1 => :w2])

println("Serial frame:     ", is_valid_on_frame(serial_frame, schema_d))
println("Non-serial frame: ", is_valid_on_frame(non_serial_frame, schema_d))

### Schema B: p → □◇p corresponds to Symmetry (Proposition 2.9)

If you can go back wherever you came from, then if $p$ is true here,
it's necessarily possible (every accessible world can see back to where $p$ holds).

In [ ]:
schema_b = Implies(p, □(◇(p)))
println("Schema B: ", schema_b)

symmetric_frame = KripkeFrame([:w1, :w2],
    [:w1 => :w2, :w2 => :w1, :w1 => :w1, :w2 => :w2])
non_symmetric_frame = KripkeFrame([:w1, :w2], [:w1 => :w2, :w2 => :w2])

println("Symmetric frame:     ", is_valid_on_frame(symmetric_frame, schema_b))
println("Non-symmetric frame: ", is_valid_on_frame(non_symmetric_frame, schema_b))

### Schema 4: □p → □□p corresponds to Transitivity (Proposition 2.11)

If accessibility chains compose, then knowing something is necessary
means knowing it's necessarily necessary — you can't "escape" necessity
by going further.

In [ ]:
schema_4 = Implies(□(p), □(□(p)))
println("Schema 4: ", schema_4)

transitive_frame = KripkeFrame([:w1, :w2, :w3],
    [:w1 => :w2, :w2 => :w3, :w1 => :w3])
non_transitive_frame = KripkeFrame([:w1, :w2, :w3],
    [:w1 => :w2, :w2 => :w3])

println("Transitive frame:     ", is_valid_on_frame(transitive_frame, schema_4))
println("Non-transitive frame: ", is_valid_on_frame(non_transitive_frame, schema_4))

### Schema 5: ◇p → □◇p corresponds to Euclideanness (Proposition 2.13)

If all successors of a world can see each other, then if something is
possible, it's necessarily possible — every accessible world agrees on
what's possible.

In [ ]:
schema_5 = Implies(◇(p), □(◇(p)))
println("Schema 5: ", schema_5)

euclidean_frame = KripkeFrame([:w1, :w2, :w3],
    [:w1 => :w2, :w1 => :w3, :w2 => :w2, :w2 => :w3, :w3 => :w2, :w3 => :w3])
non_euclidean_frame = KripkeFrame([:w1, :w2, :w3],
    [:w1 => :w2, :w1 => :w3])

println("Euclidean frame:     ", is_valid_on_frame(euclidean_frame, schema_5))
println("Non-euclidean frame: ", is_valid_on_frame(non_euclidean_frame, schema_5))

## 2.4 Normal Modal Logics

Combining these schemas gives named systems of modal logic:

| System | Axioms | Frame Class |
|:-------|:-------|:------------|
| **K** | K | All frames |
| **T** | K + T | Reflexive frames |
| **K4** | K + 4 | Transitive frames |
| **S4** | K + T + 4 | Reflexive + transitive (preorders) |
| **S5** | K + T + 5 | Reflexive + euclidean (equivalence relations) |
| **KD** | K + D | Serial frames |

Let's verify that S4 frames (preorders) validate both T and 4:

In [ ]:
# An S4 frame: reflexive and transitive
s4_frame = KripkeFrame([:w1, :w2, :w3],
    [:w1 => :w1, :w2 => :w2, :w3 => :w3,
     :w1 => :w2, :w2 => :w3, :w1 => :w3])

@assert is_reflexive(s4_frame) && is_transitive(s4_frame)

println("S4 frame validates T: ", is_valid_on_frame(s4_frame, schema_t))
println("S4 frame validates 4: ", is_valid_on_frame(s4_frame, schema_4))

And that S5 frames (equivalence relations) validate T, B, 4, and 5:

In [ ]:
# An S5 frame: equivalence relation
s5_frame = KripkeFrame([:w1, :w2],
    [:w1 => :w1, :w2 => :w2, :w1 => :w2, :w2 => :w1])

@assert is_reflexive(s5_frame) && is_symmetric(s5_frame) && is_transitive(s5_frame)

println("S5 frame validates T: ", is_valid_on_frame(s5_frame, schema_t))
println("S5 frame validates B: ", is_valid_on_frame(s5_frame, schema_b))
println("S5 frame validates 4: ", is_valid_on_frame(s5_frame, schema_4))
println("S5 frame validates 5: ", is_valid_on_frame(s5_frame, schema_5))

## Exploring on Your Own

Try these exercises:

- Build a frame that is transitive but not reflexive (K4 but not S4) and verify that schema 4 is valid but T is not
- Check whether schema D is valid on all reflexive frames (it should be — reflexivity implies seriality)
- Construct an S5 frame with 3 worlds and verify all schemas hold

In [ ]:
# Scratch space for exercises